# Limpeza dos dados

Este notebook lê `Dados_REG.tab` e gera `Dados_REG_limpo.tab` ao ser executado na pasta do projeto. Esta etapa padroniza a escrita dos dados, registra os problemas encontrados e valida a cópia antes da exportação. Codificação, imputação e escala serão definidas na etapa de transformação.

## Leitura e rastreabilidade

A leitura como texto conserva os valores originais para a auditoria, inclusive zeros, códigos, marcações de ausência e o erro de fórmula.

In [6]:
from pathlib import Path
from tempfile import NamedTemporaryFile

import pandas as pd

origem = Path("Dados_REG.tab")
destino = Path("Dados_REG_limpo.tab")
if origem.resolve() == destino.resolve():
    raise ValueError("A cópia limpa deve ter um caminho diferente do arquivo original.")

opcoes_leitura = dict(sep="\t", encoding="utf-8-sig", dtype="string", keep_default_na=False)
dados_brutos = pd.read_csv(origem, **opcoes_leitura)
dados_brutos.index = pd.RangeIndex(1, len(dados_brutos) + 1, name="linha_origem")
dados_limpos = dados_brutos.copy()

id_coluna = "n°"
morfologia_codigos = ["Cores", "Borda", "Textura", "Forma", "Brilho", "Muco", "Relevo"]
atividades = [f"Atividade {numero:02d}" for numero in range(1, 24)]
print(f"Origem: {len(dados_brutos)} registros e {len(dados_brutos.columns)} colunas.")

Origem: 506 registros e 33 colunas.


## 2. Regras aplicadas

| Caso | Tratamento | Justificativa |
|---|---|---|
| Espaços e separador `&` | Remover espaços externos e escrever o separador como ` & `. | Padronizar a grafia mantendo a ordem de predominância dos códigos. |
| `NA` e campos vazios | Representar como ausentes no pandas e exportar como campos vazios. | São resultados ausentes no experimento, distintos de um valor observado igual a zero. |
| `#NÚM!` na Atividade 23 | Marcar como ausente e guardar a grafia original na auditoria. | O resíduo de fórmula não é uma medida do ensaio. |
| `POSITIVO` e `Positiva` em GRAM | Usar `Positivo`. | São variantes de escrita da mesma categoria. |
| `Variável` e `Levedura? não` em GRAM | Usar `Desconhecido` e registrar os valores originais. | As anotações são ambíguas para a classificação em `Positivo`, `Negativo` ou `Levedura`. |

A auditoria registra as mudanças visíveis na exportação. Células originalmente vazias continuam vazias no arquivo gerado e entram no resumo de ausências.

In [7]:
alteracoes = []
campos_auditoria = ["linha_origem", "n°", "Coluna", "Original", "Limpo", "Motivo"]

def aplicar_ajuste(coluna, novos_valores, motivo):
    valores_antes = dados_limpos[coluna].fillna("")
    valores_depois = novos_valores.fillna("")
    for linha in valores_antes.index[valores_antes.ne(valores_depois)]:
        alteracoes.append([
            linha, dados_brutos.at[linha, id_coluna], coluna,
            valores_antes.at[linha], valores_depois.at[linha], motivo,
        ])
    dados_limpos[coluna] = novos_valores

for coluna in dados_limpos.columns:
    valores = dados_limpos[coluna].str.strip()
    aplicar_ajuste(coluna, valores, "Remoção de espaços externos")
    aplicar_ajuste(coluna, valores.mask(valores.isin(["", "NA"])), "Ausência no experimento")

atividade_23 = dados_limpos["Atividade 23"]
aplicar_ajuste(
    "Atividade 23", atividade_23.replace("#NÚM!", pd.NA), "Erro de fórmula na origem"
)

gram = dados_limpos["GRAM"].replace({"POSITIVO": "Positivo", "Positiva": "Positivo"})
aplicar_ajuste("GRAM", gram, "Padronização da grafia de Positivo")
aplicar_ajuste(
    "GRAM",
    gram.replace({"Variável": "Desconhecido", "Levedura? não": "Desconhecido"}),
    "Anotação ambígua de GRAM",
)

for coluna in morfologia_codigos:
    valores = dados_limpos[coluna].str.replace(r"\s*&\s*", " & ", regex=True)
    aplicar_ajuste(coluna, valores, "Padronização do separador, mantendo a ordem")

In [8]:
auditoria = pd.DataFrame(alteracoes, columns=campos_auditoria)
auditoria = auditoria.sort_values(["linha_origem", "Coluna"], kind="stable")
# As aspas tornam visíveis células vazias e espaços finais.
auditoria_exibida = auditoria.assign(
    Original=auditoria["Original"].map(repr), Limpo=auditoria["Limpo"].map(repr)
)
print("Histórico das alterações:")
print(auditoria_exibida.to_string(index=False))

ausencias = dados_limpos.isna().sum()
print(f"\nAusências no arquivo limpo: {ausencias.sum()} células")
print(ausencias[ausencias.gt(0)].to_string())

print("\nFrequência de GRAM após a limpeza:")
print(dados_limpos["GRAM"].value_counts(dropna=False).to_string())

Histórico das alterações:
 linha_origem   n°       Coluna         Original           Limpo                             Motivo
          194 1633         GRAM       'POSITIVO'      'Positivo' Padronização da grafia de Positivo
          247 1688        Borda             'NA'              ''            Ausência no experimento
          247 1688       Brilho             'NA'              ''            Ausência no experimento
          247 1688        Forma             'NA'              ''            Ausência no experimento
          247 1688         Muco             'NA'              ''            Ausência no experimento
          247 1688       Relevo             'NA'              ''            Ausência no experimento
          247 1688      Textura             'NA'              ''            Ausência no experimento
          276 1718 Atividade 23          '#NÚM!'              ''          Erro de fórmula na origem
          281 1723         GRAM       'Variável'  'Desconhecido'          

## Validação e casos preservados

Os dois registros com ID `1551` têm resultados de ensaios diferentes. Ambos ficam na cópia e aparecem com suas posições no arquivo de origem na auditoria. Valores numéricos negativos e extremos também são preservados porque a magnitude, por si só, não confirma um erro de medição.

A validação verifica o formato dos códigos, a concordância entre `Cor` e a quantidade de códigos em `Cores` e os domínios das atividades binárias e das atividades com níveis de 0 a 3. Se `OBS` ou `20` aparecerem em `Cores` numa versão futura, a exportação será interrompida para que a correção seja confirmada com a fonte.

In [9]:
ids_repetidos = dados_limpos[id_coluna].duplicated(keep=False)
print("Registros com ID repetido:")
print(dados_limpos.loc[ids_repetidos, [id_coluna, "GRAM", "Atividade 05", "Atividade 23"]])
print(f"\nLinhas inteiramente duplicadas: {dados_brutos.duplicated().sum()}")

def validar_registros(mascara, problema):
    linhas = mascara.index[mascara.fillna(False)]
    if len(linhas):
        raise ValueError(f"{problema}. Linhas de origem: {linhas.tolist()}")

cores = dados_limpos["Cores"]
codigos_reportados = cores.str.upper().str.contains(
    r"(?:^| & )(?:OBS|20)(?: & |$)", na=False
)
validar_registros(codigos_reportados, "Código OBS ou 20 em Cores; confirmar correção com a fonte")

for coluna in morfologia_codigos:
    valores = dados_limpos[coluna]
    formato_valido = valores.str.fullmatch(r"[0-9]+(?: & [0-9]+)*")
    validar_registros(valores.notna() & ~formato_valido, f"Código inválido em {coluna}")

cor = dados_limpos["Cor"]
validar_registros(cor.notna() & ~cor.str.fullmatch(r"[0-9]+"), "Contagem inválida em Cor")
quantidade_codigos = cores.str.split(" & ").str.len()
divergencias = pd.to_numeric(cor).ne(quantidade_codigos)
validar_registros(cor.notna() & cores.notna() & divergencias, "Divergência entre Cor e Cores")

dominios = {
    "GRAM": ["Negativo", "Positivo", "Levedura", "Desconhecido"],
    "Atividade 01": ["0", "1"],
}
binarias_textuais = [f"Atividade {numero:02d}" for numero in (3, 4, 16)]
dominios.update(dict.fromkeys(binarias_textuais, ["Negativo", "Positivo"]))
atividades_com_niveis = [f"Atividade {numero:02d}" for numero in (13, *range(17, 23))]
dominios.update(dict.fromkeys(atividades_com_niveis, ["0", "1", "2", "3"]))

for coluna, dominio in dominios.items():
    valores = dados_limpos[coluna]
    validar_registros(valores.notna() & ~valores.isin(dominio), f"Valor fora do domínio em {coluna}")

for coluna in atividades:
    if coluna not in binarias_textuais:
        valores = dados_limpos[coluna]
        validos = pd.to_numeric(valores, errors="coerce").notna()
        validar_registros(valores.notna() & ~validos, f"Valor não numérico em {coluna}")

ensaios_esperados = dados_brutos[atividades].apply(lambda coluna: coluna.str.strip())
ensaios_esperados = ensaios_esperados.replace("NA", "")
ensaios_esperados["Atividade 23"] = ensaios_esperados["Atividade 23"].replace("#NÚM!", "")
pd.testing.assert_frame_equal(ensaios_esperados, dados_limpos[atividades].fillna(""))
print("\nValidação concluída. Os resultados observados dos 23 ensaios foram mantidos.")

Registros com ID repetido:
                n°      GRAM Atividade 05 Atividade 23
linha_origem                                          
112           1551  Negativo         1.67        1.654
113           1551  Negativo            1        1.907

Linhas inteiramente duplicadas: 0

Validação concluída. Os resultados observados dos 23 ensaios foram mantidos.


## Exportação

O arquivo limpo usa tabulação e UTF-8 com BOM, como a origem. A saída usa as 33 colunas da origem, na mesma ordem, e escreve os campos ausentes como vazios. A saída é relida e comparada com o dataframe antes de substituir a cópia gerada em uma execução anterior.

In [10]:
exportacao = dados_limpos.reset_index(drop=True)
assert exportacao.columns.equals(dados_brutos.columns)
assert len(exportacao) == len(dados_brutos)
pd.testing.assert_series_equal(exportacao[id_coluna], dados_brutos[id_coluna].reset_index(drop=True))

with NamedTemporaryFile(dir=destino.parent, prefix=f".{destino.stem}-", delete=False) as temporario:
    caminho_temporario = Path(temporario.name)

try:
    exportacao.to_csv(caminho_temporario, sep="\t", encoding="utf-8-sig", index=False, na_rep="")
    releitura = pd.read_csv(caminho_temporario, **opcoes_leitura)
    esperado = exportacao.astype("string").fillna("")
    pd.testing.assert_frame_equal(releitura, esperado)
    caminho_temporario.replace(destino)
finally:
    caminho_temporario.unlink(missing_ok=True)

print(f"{destino.name} gerado e conferido: {len(exportacao)} registros e {len(exportacao.columns)} colunas.")

Dados_REG_limpo.tab gerado e conferido: 506 registros e 33 colunas.
